# 02 Segment-Gated Activation H02

H01 rejected global activation-aware training, but it did not kill the activation idea. The treatment helped slow / medium / late-activation slices while hurting fast and high-volume series.

H02 narrows the policy:

> Apply activation-aware training only to late-activation and slower/intermittent SKU-store series, while keeping canonical training for fast/high-volume series.

This notebook starts the H02 path. It should be used before creating the next executable ShelfOps spec so the next run has a clear rationale and acceptance criteria.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 80)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "manual_ds":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

H01B_PATH = PROJECT_ROOT / "backend" / "reports" / "experiments" / "22aedb84-a49b-499f-a6a4-f0b071c312fd.report.json"
with H01B_PATH.open() as f:
    h01b = json.load(f)


## Evidence From H01

I want H02 to come directly from the observed failure mode, not from a new guess. The first check is to restate where H01 helped and where it hurt.

In [ ]:
def segment_comparison(report: dict) -> pd.DataFrame:
    baseline = report["baseline"].get("segment_metrics") or {}
    challenger = report["challenger"].get("segment_metrics") or {}
    rows = []
    for segment in sorted(set(baseline) | set(challenger)):
        b_payload = baseline.get(segment) or {}
        c_payload = challenger.get(segment) or {}
        b_metrics = b_payload.get("metrics") or {}
        c_metrics = c_payload.get("metrics") or {}
        b_wape = b_metrics.get("wape")
        c_wape = c_metrics.get("wape")
        rows.append({
            "segment": segment,
            "rows": c_payload.get("sample_rows") or b_payload.get("sample_rows"),
            "champion_wape": b_wape,
            "challenger_wape": c_wape,
            "wape_delta": None if b_wape is None or c_wape is None else c_wape - b_wape,
            "champion_bias": b_metrics.get("bias_pct"),
            "challenger_bias": c_metrics.get("bias_pct"),
            "champion_overstock_rate": b_metrics.get("overstock_rate"),
            "challenger_overstock_rate": c_metrics.get("overstock_rate"),
        })
    return pd.DataFrame(rows)

segments = segment_comparison(h01b)
segments.sort_values("wape_delta", na_position="last")


In [ ]:
slice_rows = []
for name, payload in (h01b.get("evaluation_slices") or {}).items():
    b = payload.get("baseline") or {}
    c = payload.get("challenger") or {}
    slice_rows.append({
        "slice": name,
        "rows": payload.get("rows"),
        "champion_wape": b.get("wape"),
        "challenger_wape": c.get("wape"),
        "wape_delta": None if b.get("wape") is None or c.get("wape") is None else c.get("wape") - b.get("wape"),
        "champion_bias": b.get("bias_pct"),
        "challenger_bias": c.get("bias_pct"),
    })

pd.DataFrame(slice_rows).sort_values("wape_delta", na_position="last")


## Proposed Gating Rule

Initial H02 candidate rule:

- Use activation-aware training for slow, medium, intermittent, or late-activation SKU-store series.
- Keep canonical training for fast and high-volume SKU-store series.
- Keep the same active-holdout primary evaluation and canonical-holdout guardrail.

This should reduce the broad upward bias that hurt fast/high-volume items while preserving the useful signal for the slices where H01 helped.

## Success Criteria

H02 should not be promoted only because it helps one slice. It needs to clear the retail tradeoff:

- Slow and late-activation WAPE should improve versus champion.
- Aggregate WAPE/MASE should not regress beyond existing gates.
- Overstock dollars and combined cost proxy should not increase.
- Stockout opportunity cost can improve, but not by simply pushing more inventory everywhere.
- Promotion still requires measured pilot outcomes, so this remains benchmark/shadow evidence.

## Platform Work Required Before Running

The current ShelfOps spec system supports global activation-aware training. H02 needs a new executable spec that can gate the activation treatment by segment. The next engineering step is to add something like `m5_segment_gated_activation_window_v1` to the backend spec registry and runner.

After that spec exists, run it through ShelfOps as a new manual DS experiment with the same full-history scope used in H01b:

- Max rows: 120000
- Max series: 60
- Mode: Quick Screen
- Calibration days: 28
- Holdout days: 28


## H02 Run Outcome

I ran H02 through ShelfOps as `m5_segment_gated_activation_h02`.

The gating mechanism worked as designed: it did not simply rerun H01. H02 excluded 6,209 of the 9,856 candidate pre-activation training rows and kept 3,647 protected fast-mover pre-activation rows in training. That gives us a clean implementation read before we even judge the model metrics.

The model result is mixed. H02 is better than global activation filtering, but it is still not a champion.


In [ ]:
H02_REPORT_PATH = PROJECT_ROOT / "backend" / "reports" / "experiments" / "e57a6c4b-a54e-4025-8350-afdb1da7f859.report.json"
with H02_REPORT_PATH.open() as f:
    h02 = json.load(f)

activation = h02["activation_policy"]
{
    "policy_type": activation["policy_type"],
    "training_rows_excluded": activation["training_rows_excluded"],
    "candidate_pre_activation_training_rows": activation["candidate_pre_activation_training_rows"],
    "protected_pre_activation_training_rows": activation["protected_pre_activation_training_rows"],
    "eligible_series": activation["activation_gate_summary"]["eligible_series"],
    "protected_series": activation["activation_gate_summary"]["protected_series"],
}


In [ ]:
def primary_metric_table(report: dict) -> pd.DataFrame:
    champion = report["baseline"]["holdout_metrics"]
    challenger = report["challenger"]["holdout_metrics"]
    rows = []
    for metric in [
        "wape",
        "mase",
        "mae",
        "bias_pct",
        "coverage",
        "lost_sales_qty",
        "opportunity_cost_stockout",
        "overstock_rate",
        "overstock_dollars",
        "opportunity_cost_overstock",
    ]:
        base = champion.get(metric)
        cand = challenger.get(metric)
        rows.append({
            "metric": metric,
            "champion": base,
            "h02_challenger": cand,
            "absolute_delta": None if base is None or cand is None else cand - base,
            "pct_delta": None if base in (None, 0) or cand is None else (cand - base) / base,
        })
    return pd.DataFrame(rows)

primary_metric_table(h02)


In [ ]:
segment_comparison(h02).query("segment in ['fast', 'high_volume', 'medium', 'slow', 'intermittent']")


In [ ]:
decision_rows = []
for metric in ["service_level", "stockout_days", "lost_sales_proxy", "lost_sales_units", "overstock_dollars", "combined_cost_proxy", "po_count"]:
    champion = h02["decision_replay"]["results"]["champion"].get(metric)
    challenger = h02["decision_replay"]["results"]["challenger"].get(metric)
    decision_rows.append({
        "metric": metric,
        "champion_policy": champion,
        "h02_policy": challenger,
        "absolute_delta": None if champion is None or challenger is None else challenger - champion,
        "pct_delta": None if champion in (None, 0) or challenger is None else (challenger - champion) / champion,
    })

pd.DataFrame(decision_rows)


## Senior DS Read After H02

This is a good experiment even though it is not a winning challenger.

The positive read: H02 recovered aggregate WAPE and MASE slightly, reduced lost-sales quantity by about 4%, and reduced stockout opportunity cost by about 2%. The slow and medium segment signal from H01 also repeated, which makes the activation/ranging idea more credible.

The negative read: bias worsened, interval coverage fell, and overstock dollars still increased. Fast and high-volume series also still got worse even though their pre-activation rows were protected. That is the most important learning.

My interpretation: the issue is no longer just the data window. Because this is still one global LightGBM model plus shared calibration, changing the training treatment for eligible series can still perturb protected series through shared tree splits, category/store/product effects, and calibration behavior.

So H03 should not be another row-filtering variant. The next hypothesis should be a routed policy: use the activation-aware challenger only where it has demonstrated benefit, and keep champion predictions for protected fast/high-volume series. Then calibrate by routed segment with tighter over-forecast guardrails.


## Later Update: H03 Closed The Activation Track

H02 led naturally to H03 because fast/high-volume series still worsened even though their pre-activation rows were protected. H03 changed the lever from training-row gating to served-forecast routing: eligible slow/medium/late-activation rows received the activation challenger, while protected fast movers received the champion forecast.

H03 was the cleanest activation result:

- WAPE improved by about 1.4%.
- MASE improved by about 1.4%.
- Bias and coverage improved.
- Overstock dollars improved by about 7.2%.
- Fast/high-volume behavior was protected.

But it still failed as a champion because lost-sales quantity, stockout opportunity cost, service level, and combined cost proxy worsened. That means activation did not solve the full retail decision problem; it moved the service-level/overstock tradeoff.

Decision: do not run another activation-window variant as H04. The next manual DS hypothesis should come from a new EDA-backed family, with price/promotion proxy features as the leading candidate.
